# Dataset statistics
We will now create some simple statistics of our dataset to get a better understanding of the data we are working with. We will look at the number of publications, the number of references, the number of authors, the number of unique words in the titles and abstracts, the number of unique keywords and the number of unique journals.

## Imports

In [7]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

## Load data

In [8]:
# Load the DataFrame
df = pd.read_json('2_data/queried_publications_lang_translated_openai_tokenized.json', lines=True)

## Number of publications
We will start our light dataset analysis by looking at the number of publications per Year by publications type, to get a overall idea of how the data is distributed.

In [9]:
#| column: page
#| label: number of publications per year by publication type

# Group by Year and Category, and get the count
data_by_year_category = df.groupby(['Year', 'Category']).size().reset_index(name='Count')

# Pivot the table to have categories as columns
data_pivot = data_by_year_category.pivot_table(index='Year', columns='Category', values='Count', fill_value=0).reset_index()

# Step 1: Calculate the total number of publications per year
data_pivot['Total'] = data_pivot.drop(columns='Year').sum(axis=1)

# Step 2: Melt the data for easier plotting with Plotly, including the total count
data_melted = data_pivot.melt(id_vars=['Year', 'Total'], var_name='Category', value_name='Count')

# Step 3: Create the stacked bar chart with updated hover template
fig_combined = px.bar(data_melted, x='Year', y='Count', color='Category', 
                      title='Number of Publications per Year by Publication Type',
                      labels={'Count': 'Number of Publications', 'Year': 'Year', 'Type': 'Category'},
                      barmode='stack',
                      color_discrete_sequence=px.colors.qualitative.D3)

# Update the hover template to include the total number of publications and correct category
fig_combined.update_traces(hovertemplate='<b>Year: %{x}</b><br>Type: %{fullData.name}<br>Number of Publications: %{y}<br>Total Publications: %{customdata[0]}<extra></extra>',
                           customdata=data_melted[['Total']],
                          marker=dict(line=dict(color='grey', width=0)))

# Update layout for better visualization
fig_combined.update_layout(xaxis=dict(tickmode='linear', tick0=data_pivot['Year'].min(), dtick=2),
                           yaxis_title='Number of Publications',
                           xaxis_title='Year',
                           legend_title='Type')

fig_combined.show()

## Distribution of Publications by Journal, Author and Citations
We will now look at the distribution of our data by Journal, Author and Citations, for the top 20 values of each category. THis gives us a sense of who are the most cited authors and publications, who is most productive and in which Journal are most of the publications we are looking at published.

In [10]:
#| column: page
#| label: distribution of publications by journal, author, and number of citations

# Distribution of Publications by Journal (Top 20)
pubs_per_journal = df['Journal'].value_counts().head(20)
pubs_per_journal_df = pd.DataFrame({'Journal': pubs_per_journal.index, 'count': pubs_per_journal.values})

# Use the existing column with abbreviated journal names
pubs_per_journal_df = pubs_per_journal_df.merge(df[['Journal', 'Journal Abbreviation']].drop_duplicates(), on='Journal', how='left')

# Sort the DataFrame for proper ordering in the plot
pubs_per_journal_df = pubs_per_journal_df.sort_values('count', ascending=True)

# Split authors and create a new DataFrame
all_authors = df['Author'].str.split(',').explode().str.strip()
author_counts = all_authors.value_counts().head(20)
top_authors_df = pd.DataFrame({'Author': author_counts.index, 'count': author_counts.values})

# Sort the DataFrame for proper ordering in the plot
top_authors_df = top_authors_df.sort_values('count', ascending=True)

# Distribution of Publications by Number of Citations (Top 20)
top_cited_pubs = df.nlargest(20, 'Citation Count')
top_cited_pubs_df = pd.DataFrame({
    'Bibcode': top_cited_pubs['Bibcode'],
    'Title': top_cited_pubs['Title'],
    'count': top_cited_pubs['Citation Count'],
    'Author': top_cited_pubs['Author'],
    'Year': top_cited_pubs['Year'],
    'Journal': top_cited_pubs['Journal'],
    'Issue': top_cited_pubs['Issue'],
    'Volume': top_cited_pubs['Volume'],
    'First Page': top_cited_pubs['First Page'],
    'Last Page': top_cited_pubs['Last Page'],
    'Category': top_cited_pubs['Category'],
    'DOI': top_cited_pubs['DOI']
})

# Extract the first author
top_cited_pubs_df['First Author'] = top_cited_pubs_df['Author'].str.split(',').str[0]

# Split authors and create a new DataFrame for citations
author_citations = df[['Author', 'Citation Count']].dropna()
author_citations = author_citations.assign(Author=author_citations['Author'].str.split(',')).explode('Author').reset_index(drop=True)
author_citations['Author'] = author_citations['Author'].str.strip()
author_citations = author_citations.groupby('Author').sum().reset_index()
top_cited_authors = author_citations.nlargest(20, 'Citation Count')
top_cited_authors_df = pd.DataFrame({'Author': top_cited_authors['Author'], 'count': top_cited_authors['Citation Count']})

# Sort the DataFrames for proper ordering in the plot
top_cited_pubs_df = top_cited_pubs_df.sort_values('count', ascending=True)
top_cited_authors_df = top_cited_authors_df.sort_values('count', ascending=True)

# Create subplots
fig = make_subplots(
    rows=2, cols=2, 
    subplot_titles=('Top 20 Journals by Number of Publications', 'Top 20 Authors by Number of Publications', 'Top 20 Publications by Number of Citations', 'Top 20 Authors by Number of Citations'),
    vertical_spacing=0.1  # Reduce vertical spacing
)

# Add the first bar plot (Journals)
fig.add_trace(go.Bar(
    x=pubs_per_journal_df['count'],
    y=pubs_per_journal_df['Journal Abbreviation'],
    orientation='h',
    marker=dict(color='#3366CC', line=dict(color='grey', width=0)),  # Farbe für den ersten Plot
    name='Journals',
    customdata=pubs_per_journal_df['Journal'],
    hovertemplate='<b>%{customdata}</b><br>Number of Publications: %{x}<extra></extra>'
), row=1, col=1)

# Add the second bar plot (Authors by Publications)
fig.add_trace(go.Bar(
    x=top_authors_df['count'],
    y=top_authors_df['Author'],
    orientation='h',
    marker=dict(color='#DC3912', line=dict(color='grey', width=0)),  # Farbe für den zweiten Plot
    name='Authors by Publications',
    hovertemplate='<b>%{y}</b><br>Number of Publications: %{x}<extra></extra>'
), row=1, col=2)

# Add the third bar plot (Publications by Citations)
fig.add_trace(go.Bar(
    x=top_cited_pubs_df['count'],
    y=top_cited_pubs_df['Bibcode'],
    orientation='h',
    marker=dict(color='#FF9900', line=dict(color='grey', width=0)),  # Farbe für den dritten Plot
    name='Publications by Citations',
    customdata=top_cited_pubs_df[['Title', 'First Author', 'Year', 'Journal', 'Issue', 'Volume', 'First Page', 'Last Page', 'Category', 'DOI']],
    hovertemplate=('<b>%{customdata[0]}</b><br>Author: %{customdata[1]}<br>Year: %{customdata[2]}<br>'
                   'Category: %{customdata[8]}<br>Journal: %{customdata[3]}<br>Issue: %{customdata[4]}<br>Volume: %{customdata[5]}<br>'
                   'Pages: %{customdata[6]}-%{customdata[7]}<br>DOI: %{customdata[9]}<br>'
                   'Number of Citations: %{x}<extra></extra>')
), row=2, col=1)

# Add the fourth bar plot (Authors by Citations)
fig.add_trace(go.Bar(
    x=top_cited_authors_df['count'],
    y=top_cited_authors_df['Author'],
    orientation='h',
    marker=dict(color='#109618', line=dict(color='grey', width=0)),  # Farbe für den vierten Plot
    name='Authors by Citations',
    hovertemplate='<b>%{y}</b><br>Number of Citations: %{x}<extra></extra>'
), row=2, col=2)

# Update layout for better visualization
fig.update_layout(
    height=1200,  # Erhöhe die Höhe des Plots, um mehr Platz für die Subplots zu schaffen
    showlegend=False
)

# Update xaxis and yaxis labels
fig.update_xaxes(title_text="Number of Publications", row=1, col=1)
fig.update_yaxes(title_text="Journal", row=1, col=1)
fig.update_xaxes(title_text="Number of Publications", row=1, col=2)
fig.update_yaxes(title_text="Author", row=1, col=2)
fig.update_xaxes(title_text="Number of Citations", row=2, col=1)
fig.update_yaxes(title_text="Bibcode", row=2, col=1, showticklabels=True)
fig.update_xaxes(title_text="Number of Citations", row=2, col=2)
fig.update_yaxes(title_text="Author", row=2, col=2, showticklabels=True)

fig.show()

## Missing information

In [11]:
queried_publications = pd.read_json('2_data/queried_publications_lang_translated_openai_tokenized.json', lines=True)
print(queried_publications[queried_publications['References'].apply(lambda x: len(x) == 0)].shape[0])
print(queried_publications[queried_publications['Abstract'].apply(lambda x: len(x) == 0)].shape[0])
print(queried_publications[queried_publications['Keywords'].apply(lambda x: len(x) == 0)].shape[0])
print(queried_publications[queried_publications['Affiliation'].apply(lambda x: len(x) == 0)].shape[0])

72203
34083


TypeError: object of type 'NoneType' has no len()